In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

Machine Learning and law operate according to strikingly similar principles: they both look to historical examples inorder to infer rules to apply to new situations

In [2]:

!pip -q install transformers accelerate torch spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 8.0 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


spaCy's english small model includes NER and sentence boundary support. Doc.sents for sentence segmentation

In [3]:
from transformers import pipeline
import spacy

nli = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0   # use -1 if no GPU
)

nlp = spacy.load("en_core_web_sm")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Standard ready-made nli model commonly used as a zero shot classifier by rewriting labels as hypotheses

In [4]:
examples = [
    "The government failed to create jobs despite repeated promises.",
    "Rahul Gandhi presented a strong critique of the budget.",
    "The article reports that Modi and Gandhi addressed separate rallies.",
    "The chief minister's welfare scheme was praised by party workers."
]

target = "Narendra Modi"
labels = [
    f"This text supports {target}.",
    f"This text opposes {target}.",
    f"This text is neutral toward {target}."
]

for text in examples:
    out = nli(text, labels, multi_label=False)
    print("\nTEXT:", text)
    print(list(zip(out["labels"], [round(x, 4) for x in out["scores"]])))


TEXT: The government failed to create jobs despite repeated promises.
[('This text opposes Narendra Modi.', 0.622), ('This text is neutral toward Narendra Modi.', 0.213), ('This text supports Narendra Modi.', 0.165)]

TEXT: Rahul Gandhi presented a strong critique of the budget.
[('This text opposes Narendra Modi.', 0.8186), ('This text supports Narendra Modi.', 0.0985), ('This text is neutral toward Narendra Modi.', 0.0829)]

TEXT: The article reports that Modi and Gandhi addressed separate rallies.
[('This text supports Narendra Modi.', 0.7381), ('This text is neutral toward Narendra Modi.', 0.1644), ('This text opposes Narendra Modi.', 0.0975)]

TEXT: The chief minister's welfare scheme was praised by party workers.
[('This text supports Narendra Modi.', 0.6396), ('This text is neutral toward Narendra Modi.', 0.2595), ('This text opposes Narendra Modi.', 0.1009)]


In [5]:
templates = {
    "favour": "This text supports {}.",
    "against": "This text opposes {}.",
    "neutral": "This text is neutral toward {}."
} # vs author supports etc

In [6]:
def split_sentences(text, nlp):
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents if sent.text.strip()]

def score_sentence_for_target(sentence, target, clf):
    labels = [
        f"This text supports {target}.",
        f"This text opposes {target}.",
        f"This text is neutral toward {target}."
    ]
    out = clf(sentence, labels, multi_label=False)
    score_map = dict(zip(out["labels"], out["scores"]))
    return score_map

def get_target_evidence(article_text, target, clf, nlp, top_k=3):
    sentences = split_sentences(article_text, nlp)

    scored = []
    for s in sentences:
        if target.lower() in s.lower():
            scores = score_sentence_for_target(s, target, clf)
            scored.append((s, scores))

    if not scored:
        return {
            "target": target,
            "label": "no_evidence",
            "scores": {},
            "evidence": []
        }

    support_key = f"This text supports {target}."
    oppose_key = f"This text opposes {target}."
    neutral_key = f"This text is neutral toward {target}."

    mean_scores = {
        "favour": sum(x[1][support_key] for x in scored) / len(scored),
        "against": sum(x[1][oppose_key] for x in scored) / len(scored),
        "neutral": sum(x[1][neutral_key] for x in scored) / len(scored),
    }

    final_label = max(mean_scores, key=mean_scores.get)

    if final_label == "favour":
        key = support_key
    elif final_label == "against":
        key = oppose_key
    else:
        key = neutral_key

    evidence = sorted(scored, key=lambda x: x[1][key], reverse=True)[:top_k]

    return {
        "target": target,
        "label": final_label,
        "scores": mean_scores,
        "evidence": [x[0] for x in evidence]
    }

In [7]:
article = """
Narendra Modi promised large-scale job creation, but critics say the government has failed to deliver.
Supporters argue that infrastructure spending will create long-term opportunity.
Opposition leaders accused the administration of ignoring youth unemployment.
The article also notes that Rahul Gandhi criticized the budget during a rally.
"""

print(get_target_evidence(article, "Narendra Modi", nli, nlp))
print(get_target_evidence(article, "Rahul Gandhi", nli, nlp))

{'target': 'Narendra Modi', 'label': 'against', 'scores': {'favour': 0.05516045168042183, 'against': 0.8738190531730652, 'neutral': 0.07102052867412567}, 'evidence': ['Narendra Modi promised large-scale job creation, but critics say the government has failed to deliver.']}
{'target': 'Rahul Gandhi', 'label': 'favour', 'scores': {'favour': 0.4588836133480072, 'against': 0.4288230836391449, 'neutral': 0.11229328066110611}, 'evidence': ['The article also notes that Rahul Gandhi criticized the budget during a rally.']}


https://aclanthology.org/2021.findings-acl.208.pdf P-Stance dataset

In [8]:
from dataclasses import dataclass
from typing import List, Dict, Optional

@dataclass
class TargetStanceResult:
    target: str
    label: str                  # "favor", "against", "neutral", "mixed", "no_evidence"
    scores: Dict[str, float]    # {"favor": ..., "against": ..., "neutral": ...}
    evidence: List[str]         # top supporting chunks
    num_chunks_used: int

@dataclass
class StancePipelineOutput:
    text_id: Optional[str]
    targets: List[str]
    results: List[TargetStanceResult]

example {
    "text_id": "demo_001",
    "targets": ["Narendra Modi", "Rahul Gandhi"],
    "results": [
        {
            "target": "Narendra Modi",
            "label": "against",
            "scores": {
                "favor": 0.18,
                "against": 0.72,
                "neutral": 0.10
            },
            "evidence": [
                "Critics said Modi's jobs promises have failed to materialize.",
                "The article argues that the government's record has disappointed young voters."
            ],
            "num_chunks_used": 4
        },
        {
            "target": "Rahul Gandhi",
            "label": "neutral",
            "scores": {
                "favor": 0.21,
                "against": 0.19,
                "neutral": 0.60
            },
            "evidence": [
                "Rahul Gandhi also addressed the rally later in the evening."
            ],
            "num_chunks_used": 2
        }
    ]
}

P-Stance is a good choice here because it is a political-domain stance dataset with 21,574 labeled English tweets for the targets Donald Trump, Joe Biden, and Bernie Sanders,

In [10]:
# clone the P-Stance repo into /kaggle/working and inspect files

%cd /kaggle/working

!git clone https://github.com/chuchun8/PStance.git

import os

repo_path = "/kaggle/working/PStance"

for root, dirs, files in os.walk(repo_path, topdown=True):
    level = root.replace(repo_path, "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = "    " * (level + 1)
    for f in files[:20]:
        print(f"{subindent}{f}")

/kaggle/working
Cloning into 'PStance'...
remote: Enumerating objects: 50, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 50 (delta 16), reused 4 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (50/50), 342.16 KiB | 1.75 MiB/s, done.
Resolving deltas: 100% (16/16), done.
PStance/
    LICENSE
    README.md
    requirements.txt
    .git/
        index
        packed-refs
        HEAD
        description
        config
        objects/
            info/
            pack/
                pack-5417f07ba689aac1152a03825dd4aa87f32e14cd.idx
                pack-5417f07ba689aac1152a03825dd4aa87f32e14cd.pack
        info/
            exclude
        logs/
            HEAD
            refs/
                heads/
                    main
                remotes/
                    origin/
                        HEAD
        refs/
            heads/
                main
            remotes/
                origin/
    

In [11]:
!pip -q install gdown
!gdown --folder "https://drive.google.com/drive/folders/1so8lY1XKpnhUtTvb15edEz6aeHt7CSuh" -O /kaggle/working/pstance_data

Retrieving folder contents
Processing file 1mhEGjKBJI1cH0NNthsTgDSDqFzsBGi2P raw_test_bernie.csv
Processing file 1VfwZ3BYgwdLu4tHYLn2XzNn5_YKUxrGU raw_test_biden.csv
Processing file 18wGOBjvDlna-DceQ7pqf6JYXZPiLkG1d raw_test_trump.csv
Processing file 11OPnWs_mBmsCl32VeK8Sb2aTwKIz4RdV raw_train_bernie.csv
Processing file 1Onq6oAxe--l7aI047T3d979N_JRunbVc raw_train_biden.csv
Processing file 1830SzCs-AllGAQEoMW14rAKIpLH8JBq- raw_train_trump.csv
Processing file 1068LHFvG5-zgwfoIsGpJ7v-T-xNUQo60 raw_val_bernie.csv
Processing file 1NuAM8DItJhoYdukBz9pPHW2e4EjYAd9i raw_val_biden.csv
Processing file 1xKmh65J1RbpzWb1mnIFs4i3SiAKTHT2V raw_val_trump.csv
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1mhEGjKBJI1cH0NNthsTgDSDqFzsBGi2P
To: /kaggle/working/pstance_data/raw_test_bernie.csv
100%|████████████████████████████████████████| 133k/133k [00:00<00:00, 98.4MB/s]
Downloading...
From: htt

Let's get zero shot baseline
For fine tuning a simpler one like roberta-base can be finetuned

In [12]:


import pandas as pd
import glob
import os

data_dir = "/kaggle/working/pstance_data"
files = sorted(glob.glob(os.path.join(data_dir, "*.csv")))

print("Files found:")
for f in files:
    print(" -", os.path.basename(f))

dfs = []
for f in files:
    df = pd.read_csv(f)
    df["__file__"] = os.path.basename(f)
    dfs.append(df)

full_df = pd.concat(dfs, ignore_index=True)

print("\nShape:", full_df.shape)
print("\nColumns:", full_df.columns.tolist())

print("\nFirst 5 rows:")
display(full_df.head())

print("\nNull counts:")
display(full_df.isna().sum())

for col in full_df.columns:
    print(f"\nUnique values for {col} (up to 20):")
    vals = full_df[col].dropna().astype(str).unique().tolist()
    print(vals[:20])

Files found:
 - raw_test_bernie.csv
 - raw_test_biden.csv
 - raw_test_trump.csv
 - raw_train_bernie.csv
 - raw_train_biden.csv
 - raw_train_trump.csv
 - raw_val_bernie.csv
 - raw_val_biden.csv
 - raw_val_trump.csv

Shape: (21574, 4)

Columns: ['Tweet', 'Target', 'Stance', '__file__']

First 5 rows:


,Tweet,Target,Stance,__file__
0,"#IEndorseBernie for tons of reasons, but this ...",Bernie Sanders,FAVOR,raw_test_bernie.csv
1,A big problem w/#Bernie left is not only preoc...,Bernie Sanders,AGAINST,raw_test_bernie.csv
2,"This poll is not reflecting anything: ""age was...",Bernie Sanders,AGAINST,raw_test_bernie.csv
3,So proud how @BernieSanders is shedding light ...,Bernie Sanders,FAVOR,raw_test_bernie.csv
4,"According to media bias fact checker, you have...",Bernie Sanders,FAVOR,raw_test_bernie.csv



Null counts:


Tweet       0
Target      0
Stance      0
__file__    0
dtype: int64


Unique values for Tweet (up to 20):
['#IEndorseBernie for tons of reasons, but this one is my number 1. We are running out of time to save our planet from the worst of climate change. We NEED Bernies #GreenNewDeal. #Bernie @Lafayette, Indiana', 'A big problem w/#Bernie left is not only preoccupation with #socialism v. #capitalism over Dem v. Repub as @jonathanchait said but also equating socialism with progressivism. Hence apprehension about #ElizabethWarren, blind spot to #BlackLivesMatter, etc', 'This poll is not reflecting anything: "age was lightly weighted, thus reflecting the gender and age breakdowns of Iowa Democratic caucusgoers in previous election years." #BernieSanders will turn out a huge number of young &amp; working class voters. So keep pushing Berners!', 'So proud how @BernieSanders is shedding light on who is truly a progressive and pushing the party back to left out of center. #DemDebate2020', 'According to media bias fact checker, you have high ratings for factual 

This one is binary stance and three political targets

In [13]:
# add split + normalized label, then inspect distributions

def infer_split(filename):
    filename = filename.lower()
    if "train" in filename:
        return "train"
    if "val" in filename:
        return "val"
    if "test" in filename:
        return "test"
    return "unknown"

full_df["split"] = full_df["__file__"].apply(infer_split)
full_df["label"] = full_df["Stance"].str.lower()

print("Split counts:")
display(full_df["split"].value_counts())

print("\nLabel counts overall:")
display(full_df["label"].value_counts())

print("\nCounts by split and target:")
display(pd.crosstab(full_df["split"], full_df["Target"]))

print("\nCounts by split and label:")
display(pd.crosstab(full_df["split"], full_df["label"]))

print("\nCounts by target and label:")
display(pd.crosstab(full_df["Target"], full_df["label"]))

Split counts:


split
train    17224
val       2193
test      2157
Name: count, dtype: int64


Label counts overall:


label
against    11143
favor      10431
Name: count, dtype: int64


Counts by split and target:


Target,Bernie Sanders,Donald Trump,Joe Biden
split,,,
test,635,777,745
train,5056,6362,5806
val,634,814,745



Counts by split and label:


label,against,favor
split,,
test,1125,1032
train,8877,8347
val,1141,1052



Counts by target and label:


label,against,favor
Target,,
Bernie Sanders,2774,3551
Donald Trump,4290,3663
Joe Biden,4079,3217


In [14]:
# Step 5: make a small balanced sample from the TEST split

sample_per_target_label = 25   # total sample = 3 targets * 2 labels * 25 = 150

test_df = full_df[full_df["split"] == "test"].copy()

small_test_df = (
    test_df
    .groupby(["Target", "label"], group_keys=False)
    .apply(lambda x: x.sample(n=min(sample_per_target_label, len(x)), random_state=42))
    .reset_index(drop=True)
)

print("Small test shape:", small_test_df.shape)
display(pd.crosstab(small_test_df["Target"], small_test_df["label"]))

display(small_test_df.head(10))

Small test shape: (150, 6)


/tmp/ipykernel_55/1890738539.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(sample_per_target_label, len(x)), random_state=42))


label,against,favor
Target,,
Bernie Sanders,25,25
Donald Trump,25,25
Joe Biden,25,25


,Tweet,Target,Stance,__file__,split,label
0,Not even #BernieSanders will... just watch how...,Bernie Sanders,AGAINST,raw_test_bernie.csv,test,against
1,"Bernie has hosted medium- to high-dollar, clos...",Bernie Sanders,AGAINST,raw_test_bernie.csv,test,against
2,"I respect you & your work Shaun, but please st...",Bernie Sanders,AGAINST,raw_test_bernie.csv,test,against
3,The following Dem candidates need to drop out:...,Bernie Sanders,AGAINST,raw_test_bernie.csv,test,against
4,#Socialism and #Bernie are policy failures. Th...,Bernie Sanders,AGAINST,raw_test_bernie.csv,test,against
5,You are actually correct for once. #BernieSand...,Bernie Sanders,AGAINST,raw_test_bernie.csv,test,against
6,So you want me to stop mentioning that #Bernie...,Bernie Sanders,AGAINST,raw_test_bernie.csv,test,against
7,Is bernie campaigning on his bicycle?? Or is h...,Bernie Sanders,AGAINST,raw_test_bernie.csv,test,against
8,I've never seen someone be so completely wrong...,Bernie Sanders,AGAINST,raw_test_bernie.csv,test,against
9,Ignorant radical leftist taxpayers bribed to v...,Bernie Sanders,AGAINST,raw_test_bernie.csv,test,against


In [15]:
# Step 6: binary zero-shot stance prediction for one text-target pair

def predict_stance_binary(text, target, clf):
    labels = [
        f"This text supports {target}.",
        f"This text opposes {target}."
    ]
    
    out = clf(text, labels, multi_label=False)
    score_map = dict(zip(out["labels"], out["scores"]))
    
    support_key = f"This text supports {target}."
    oppose_key = f"This text opposes {target}."
    
    support_score = score_map.get(support_key, 0.0)
    oppose_score = score_map.get(oppose_key, 0.0)
    
    pred_label = "favor" if support_score >= oppose_score else "against"
    
    return {
        "target": target,
        "pred_label": pred_label,
        "scores": {
            "favor": support_score,
            "against": oppose_score
        },
        "evidence": [text]   # for tweets, the whole tweet is the evidence
    }

# quick sanity check on 3 examples
for i in range(3):
    row = small_test_df.iloc[i]
    result = predict_stance_binary(row["Tweet"], row["Target"], nli)
    print(f"\nExample {i}")
    print("TARGET:", row["Target"])
    print("GOLD:", row["label"])
    print("PRED:", result["pred_label"])
    print("SCORES:", result["scores"])
    print("TWEET:", row["Tweet"][:250], "...")


Example 0
TARGET: Bernie Sanders
GOLD: against
PRED: against
SCORES: {'favor': 0.08875731378793716, 'against': 0.9112427234649658}
TWEET: Not even #BernieSanders will... just watch how butthurt he was after he LOST last time... a #TwoTime loser will be worse ...

Example 1
TARGET: Bernie Sanders
GOLD: against
PRED: favor
SCORES: {'favor': 0.9690670967102051, 'against': 0.030932912603020668}
TWEET: Bernie has hosted medium- to high-dollar, closed-door fundraisers in New York, Los Angeles and elsewhere from his previous campaign. Not sure if Bernie supporters are aware of that. #Bernie #Yang2020 ...

Example 2
TARGET: Bernie Sanders
GOLD: against
PRED: against
SCORES: {'favor': 0.19810952246189117, 'against': 0.80189049243927}
TWEET: I respect you & your work Shaun, but please stop sinking to low places. Support your candidate, but please stop vilifying another viable candidate. Youre acting like #BernieSanders which is not only unbecoming, but its making me rethink my overall re ...


In [17]:
# evaluate zero-shot binary stance on the small balanced test set

from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

preds = []

for _, row in tqdm(small_test_df.iterrows(), total=len(small_test_df)):
    result = predict_stance_binary(row["Tweet"], row["Target"], nli)
    preds.append(result["pred_label"])

small_test_df_eval = small_test_df.copy()
small_test_df_eval["pred_label"] = preds

acc = accuracy_score(small_test_df_eval["label"], small_test_df_eval["pred_label"])
macro_f1 = f1_score(small_test_df_eval["label"], small_test_df_eval["pred_label"], average="macro")

print("Accuracy:", round(acc, 4))
print("Macro-F1:", round(macro_f1, 4))

print("\nClassification report:")
print(classification_report(small_test_df_eval["label"], small_test_df_eval["pred_label"], digits=4))

print("\nPer-target accuracy:")
display(
    small_test_df_eval.groupby("Target")
    .apply(lambda x: (x["label"] == x["pred_label"]).mean())
    .rename("accuracy")
    .reset_index()
)

  0%|          | 0/150 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Accuracy: 0.62
Macro-F1: 0.6198

Classification report:
              precision    recall  f1-score   support

     against     0.6250    0.6000    0.6122        75
       favor     0.6154    0.6400    0.6275        75

    accuracy                         0.6200       150
   macro avg     0.6202    0.6200    0.6198       150
weighted avg     0.6202    0.6200    0.6198       150


Per-target accuracy:


/tmp/ipykernel_55/1399353276.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: (x["label"] == x["pred_label"]).mean())


,Target,accuracy
0,Bernie Sanders,0.54
1,Donald Trump,0.62
2,Joe Biden,0.70


A zero-shot NLI baseline achieved 0.62 accuracy and 0.62 macro-F1 on a balanced P-Stance sample, with substantial variation across political targets. This suggests the target-aware stance pipeline is viable, but also target-sensitive and likely to benefit from domain adaptation.

In [18]:
# Step 8: run zero-shot binary stance on the full test split

from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

full_test_df = full_df[full_df["split"] == "test"].copy()

preds = []
for _, row in tqdm(full_test_df.iterrows(), total=len(full_test_df)):
    result = predict_stance_binary(row["Tweet"], row["Target"], nli)
    preds.append(result["pred_label"])

full_test_eval = full_test_df.copy()
full_test_eval["pred_label"] = preds

acc = accuracy_score(full_test_eval["label"], full_test_eval["pred_label"])
macro_f1 = f1_score(full_test_eval["label"], full_test_eval["pred_label"], average="macro")

print("Full test Accuracy:", round(acc, 4))
print("Full test Macro-F1:", round(macro_f1, 4))

print("\nClassification report:")
print(classification_report(full_test_eval["label"], full_test_eval["pred_label"], digits=4))

print("\nPer-target accuracy:")
per_target_acc = (
    full_test_eval.groupby("Target")
    .apply(lambda x: (x["label"] == x["pred_label"]).mean())
    .rename("accuracy")
    .reset_index()
)
display(per_target_acc)

  0%|          | 0/2157 [00:00<?, ?it/s]

Full test Accuracy: 0.7065
Full test Macro-F1: 0.7059

Classification report:
              precision    recall  f1-score   support

     against     0.7169    0.7227    0.7198      1125
       favor     0.6950    0.6890    0.6920      1032

    accuracy                         0.7065      2157
   macro avg     0.7060    0.7058    0.7059      2157
weighted avg     0.7064    0.7065    0.7065      2157


Per-target accuracy:


/tmp/ipykernel_55/64204111.py:28: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: (x["label"] == x["pred_label"]).mean())


,Target,accuracy
0,Bernie Sanders,0.681890
1,Donald Trump,0.664093
2,Joe Biden,0.771812


In [19]:
#  reusable long-text target-aware stance function with evidence

def split_sentences(text, nlp):
    doc = nlp(text)
    return [s.text.strip() for s in doc.sents if s.text.strip()]

def analyze_text_targets(
    text,
    targets,
    clf,
    nlp,
    alias_map=None,
    top_k_evidence=3,
    include_neutral=True,
    text_id=None,
):
    alias_map = alias_map or {}
    sentences = split_sentences(text, nlp)

    results = []

    for target in targets:
        aliases = alias_map.get(target, [])
        candidate_names = [target] + aliases

        matched_sentences = []
        for sent in sentences:
            sent_low = sent.lower()
            if any(name.lower() in sent_low for name in candidate_names):
                matched_sentences.append(sent)

        if not matched_sentences:
            results.append({
                "target": target,
                "label": "no_evidence",
                "scores": {},
                "evidence": [],
                "num_sentences_used": 0
            })
            continue

        scored = []
        for sent in matched_sentences:
            labels = [
                f"This text supports {target}.",
                f"This text opposes {target}.",
            ]
            if include_neutral:
                labels.append(f"This text is neutral toward {target}.")

            out = clf(sent, labels, multi_label=False)
            score_map = dict(zip(out["labels"], out["scores"]))
            scored.append((sent, score_map))

        support_key = f"This text supports {target}."
        oppose_key = f"This text opposes {target}."

        mean_scores = {
            "favor": sum(x[1].get(support_key, 0.0) for x in scored) / len(scored),
            "against": sum(x[1].get(oppose_key, 0.0) for x in scored) / len(scored),
        }

        if include_neutral:
            neutral_key = f"This text is neutral toward {target}."
            mean_scores["neutral"] = sum(x[1].get(neutral_key, 0.0) for x in scored) / len(scored)

        final_label = max(mean_scores, key=mean_scores.get)

        if final_label == "favor":
            winner_key = support_key
        elif final_label == "against":
            winner_key = oppose_key
        else:
            winner_key = neutral_key

        evidence = sorted(
            scored,
            key=lambda x: x[1].get(winner_key, 0.0),
            reverse=True
        )[:top_k_evidence]

        results.append({
            "target": target,
            "label": final_label,
            "scores": mean_scores,
            "evidence": [x[0] for x in evidence],
            "num_sentences_used": len(matched_sentences)
        })

    return {
        "text_id": text_id,
        "targets": targets,
        "results": results
    }

In [20]:
# try the pipeline on a longer article-like paragraph

sample_text = """
Narendra Modi defended the government's economic record and said that long-term infrastructure
investment would benefit young people. However, opposition leaders argued that unemployment remains
high and that the promises made in earlier campaigns have not translated into enough real jobs.
Rahul Gandhi criticized the government in a rally, saying that the youth of the country had been
let down by repeated economic mismanagement. The article also noted that BJP supporters praised
the administration's welfare delivery and argued that the opposition was ignoring broader development gains.
"""

sample_targets = ["Narendra Modi", "Rahul Gandhi", "BJP"]

out = analyze_text_targets(
    text=sample_text,
    targets=sample_targets,
    clf=nli,
    nlp=nlp,
    top_k_evidence=2,
    include_neutral=True,
    text_id="demo_long_001"
)

out

{'text_id': 'demo_long_001',
 'targets': ['Narendra Modi', 'Rahul Gandhi', 'BJP'],
 'results': [{'target': 'Narendra Modi',
   'label': 'favor',
   'scores': {'favor': 0.9816301465034485,
    'against': 0.00424707867205143,
    'neutral': 0.014122779481112957},
   'evidence': ["Narendra Modi defended the government's economic record and said that long-term infrastructure\ninvestment would benefit young people."],
   'num_sentences_used': 1},
  {'target': 'Rahul Gandhi',
   'label': 'favor',
   'scores': {'favor': 0.6178647875785828,
    'against': 0.2498801052570343,
    'neutral': 0.13225509226322174},
   'evidence': ['Rahul Gandhi criticized the government in a rally, saying that the youth of the country had been\nlet down by repeated economic mismanagement.'],
   'num_sentences_used': 1},
  {'target': 'BJP',
   'label': 'favor',
   'scores': {'favor': 0.9641126394271851,
    'against': 0.014164378866553307,
    'neutral': 0.021723031997680664},
   'evidence': ["The article also note

In [21]:
# pretty printer for results

def print_stance_output(out):
    print(f"text_id: {out.get('text_id')}")
    print("=" * 80)
    
    for res in out["results"]:
        print(f"TARGET: {res['target']}")
        print(f"  LABEL: {res['label']}")
        print(f"  SCORES: { {k: round(v, 4) for k, v in res['scores'].items()} }")
        print(f"  NUM_SENTENCES_USED: {res['num_sentences_used']}")
        print("  EVIDENCE:")
        for i, ev in enumerate(res["evidence"], 1):
            print(f"    {i}. {ev}")
        print("-" * 80)

print_stance_output(out)

text_id: demo_long_001
TARGET: Narendra Modi
  LABEL: favor
  SCORES: {'favor': 0.9816, 'against': 0.0042, 'neutral': 0.0141}
  NUM_SENTENCES_USED: 1
  EVIDENCE:
    1. Narendra Modi defended the government's economic record and said that long-term infrastructure
investment would benefit young people.
--------------------------------------------------------------------------------
TARGET: Rahul Gandhi
  LABEL: favor
  SCORES: {'favor': 0.6179, 'against': 0.2499, 'neutral': 0.1323}
  NUM_SENTENCES_USED: 1
  EVIDENCE:
    1. Rahul Gandhi criticized the government in a rally, saying that the youth of the country had been
let down by repeated economic mismanagement.
--------------------------------------------------------------------------------
TARGET: BJP
  LABEL: favor
  SCORES: {'favor': 0.9641, 'against': 0.0142, 'neutral': 0.0217}
  NUM_SENTENCES_USED: 1
  EVIDENCE:
    1. The article also noted that BJP supporters praised
the administration's welfare delivery and argued that the opp

In [22]:
#  test alias_map on a longer example

sample_text_2 = """
Prime Minister Modi said the reforms would strengthen the economy over time.
Critics argued that Modi had failed to address unemployment and rising inequality.
Congress leaders said the BJP-led government was ignoring the struggles of ordinary workers.
Supporters of the ruling party responded that the opposition was downplaying welfare delivery and infrastructure gains.
"""

sample_targets_2 = ["Narendra Modi", "BJP", "Congress"]

alias_map_2 = {
    "Narendra Modi": ["Modi", "Prime Minister Modi", "PM Modi"],
    "BJP": ["BJP-led government", "ruling party"],
    "Congress": ["Congress leaders", "the opposition"]
}

out2 = analyze_text_targets(
    text=sample_text_2,
    targets=sample_targets_2,
    clf=nli,
    nlp=nlp,
    alias_map=alias_map_2,
    top_k_evidence=2,
    include_neutral=True,
    text_id="demo_long_002"
)

print_stance_output(out2)

text_id: demo_long_002
TARGET: Narendra Modi
  LABEL: favor
  SCORES: {'favor': 0.4838, 'against': 0.4741, 'neutral': 0.0421}
  NUM_SENTENCES_USED: 2
  EVIDENCE:
    1. Prime Minister Modi said the reforms would strengthen the economy over time.
    2. Critics argued that Modi had failed to address unemployment and rising inequality.
--------------------------------------------------------------------------------
TARGET: BJP
  LABEL: against
  SCORES: {'favor': 0.3202, 'against': 0.5902, 'neutral': 0.0895}
  NUM_SENTENCES_USED: 2
  EVIDENCE:
    1. Congress leaders said the BJP-led government was ignoring the struggles of ordinary workers.
    2. Supporters of the ruling party responded that the opposition was downplaying welfare delivery and infrastructure gains.
--------------------------------------------------------------------------------
TARGET: Congress
  LABEL: favor
  SCORES: {'favor': 0.6173, 'against': 0.2948, 'neutral': 0.0879}
  NUM_SENTENCES_USED: 2
  EVIDENCE:
    1. Con

Part A: target linking

“Which sentences are about this target?”

This is where:

alias maps
NER
keyword rules
maybe fuzzy matching
maybe lightweight coreference later

matter.

Part B: stance scoring

“Given that this sentence is about the target, is it favorable, against, or neutral?”

This is where:

zero-shot NLI
or fine-tuned RoBERTa later

matters.

So no, I would not fine-tune just because alias matching is imperfect.

In [26]:
# add a mixed/uncertain rule to the long-text analyzer

def analyze_text_targets(
    text,
    targets,
    clf,
    nlp,
    alias_map=None,
    top_k_evidence=3,
    include_neutral=True,
    text_id=None,
    mixed_margin=0.08,   # if top two scores are within this gap, return "mixed"
):
    alias_map = alias_map or {}
    sentences = split_sentences(text, nlp)

    results = []

    for target in targets:
        aliases = alias_map.get(target, [])
        candidate_names = [target] + aliases

        matched_sentences = []
        for sent in sentences:
            sent_low = sent.lower()
            # if any(name.lower() in sent_low for name in candidate_names):
            if sentence_mentions_target(sent, target, aliases):
                matched_sentences.append(sent)

        if not matched_sentences:
            results.append({
                "target": target,
                "label": "no_evidence",
                "scores": {},
                "evidence": [],
                "num_sentences_used": 0
            })
            continue

        scored = []
        for sent in matched_sentences:
            labels = [
                f"This text supports {target}.",
                f"This text opposes {target}."
            ]
            if include_neutral:
                labels.append(f"This text is neutral toward {target}.")

            out = clf(sent, labels, multi_label=False)
            score_map = dict(zip(out["labels"], out["scores"]))
            scored.append((sent, score_map))

        support_key = f"This text supports {target}."
        oppose_key = f"This text opposes {target}."

        mean_scores = {
            "favor": sum(x[1].get(support_key, 0.0) for x in scored) / len(scored),
            "against": sum(x[1].get(oppose_key, 0.0) for x in scored) / len(scored),
        }

        if include_neutral:
            neutral_key = f"This text is neutral toward {target}."
            mean_scores["neutral"] = sum(x[1].get(neutral_key, 0.0) for x in scored)

            mean_scores["neutral"] /= len(scored)

        # find best and second-best labels
        sorted_scores = sorted(mean_scores.items(), key=lambda x: x[1], reverse=True)
        best_label, best_score = sorted_scores[0]
        second_label, second_score = sorted_scores[1]

        if abs(best_score - second_score) < mixed_margin:
            final_label = "mixed"
            winner_key = None
            # use strongest overall evidence among top 2 labels
            top_two = {best_label, second_label}
            evidence_scored = []
            for sent, score_map in scored:
                sent_score = 0.0
                if "favor" in top_two:
                    sent_score = max(sent_score, score_map.get(support_key, 0.0))
                if "against" in top_two:
                    sent_score = max(sent_score, score_map.get(oppose_key, 0.0))
                if include_neutral and "neutral" in top_two:
                    sent_score = max(sent_score, score_map.get(neutral_key, 0.0))
                evidence_scored.append((sent, sent_score))
            evidence = sorted(evidence_scored, key=lambda x: x[1], reverse=True)[:top_k_evidence]
            evidence_text = [x[0] for x in evidence]
        else:
            final_label = best_label
            if final_label == "favor":
                winner_key = support_key
            elif final_label == "against":
                winner_key = oppose_key
            else:
                winner_key = neutral_key

            evidence = sorted(
                scored,
                key=lambda x: x[1].get(winner_key, 0.0),
                reverse=True
            )[:top_k_evidence]
            evidence_text = [x[0] for x in evidence]

        results.append({
            "target": target,
            "label": final_label,
            "scores": mean_scores,
            "evidence": evidence_text,
            "num_sentences_used": len(matched_sentences)
        })

    return {
        "text_id": text_id,
        "targets": targets,
        "results": results
    }

In [28]:
alias_map_2 = {
    "Narendra Modi": ["Modi", "Prime Minister Modi", "PM Modi"],
    "BJP": ["BJP-led government", "ruling party"],
    "Congress": ["Congress leaders"]
}

In [29]:
out2 = analyze_text_targets(
    text=sample_text_2,
    targets=sample_targets_2,
    clf=nli,
    nlp=nlp,
    alias_map=alias_map_2,
    top_k_evidence=2,
    include_neutral=True,
    text_id="demo_long_002"
)

print_stance_output(out2)

text_id: demo_long_002
TARGET: Narendra Modi
  LABEL: mixed
  SCORES: {'favor': 0.4838, 'against': 0.4741, 'neutral': 0.0421}
  NUM_SENTENCES_USED: 2
  EVIDENCE:
    1. Prime Minister Modi said the reforms would strengthen the economy over time.
    2. Critics argued that Modi had failed to address unemployment and rising inequality.
--------------------------------------------------------------------------------
TARGET: BJP
  LABEL: against
  SCORES: {'favor': 0.3202, 'against': 0.5902, 'neutral': 0.0895}
  NUM_SENTENCES_USED: 2
  EVIDENCE:
    1. Congress leaders said the BJP-led government was ignoring the struggles of ordinary workers.
    2. Supporters of the ruling party responded that the opposition was downplaying welfare delivery and infrastructure gains.
--------------------------------------------------------------------------------
TARGET: Congress
  LABEL: favor
  SCORES: {'favor': 0.739, 'against': 0.237, 'neutral': 0.024}
  NUM_SENTENCES_USED: 1
  EVIDENCE:
    1. Congre

In [25]:
# safer target matching with regex word boundaries

import re

def compile_alias_patterns(target, aliases=None):
    aliases = aliases or []
    names = [target] + aliases
    patterns = []
    for name in names:
        # match whole phrase with word boundaries, case-insensitive
        pat = re.compile(r"\b" + re.escape(name) + r"\b", flags=re.IGNORECASE)
        patterns.append((name, pat))
    return patterns

def sentence_mentions_target(sent, target, aliases=None):
    patterns = compile_alias_patterns(target, aliases)
    for name, pat in patterns:
        if pat.search(sent):
            return True
    return False

In [30]:
#  simple NER-based target suggestion helper

def suggest_targets(text, nlp):
    doc = nlp(text)
    ents = []
    for ent in doc.ents:
        if ent.label_ in {"PERSON", "ORG"}:
            ents.append(ent.text.strip())
    # preserve order, remove duplicates
    seen = set()
    unique_ents = []
    for x in ents:
        if x.lower() not in seen:
            seen.add(x.lower())
            unique_ents.append(x)
    return unique_ents

# quick test
print(suggest_targets(sample_text_2, nlp))

['Modi', 'Congress', 'BJP']


In [31]:
#  wrapper that accepts manual targets or auto-suggests them

def run_stance_pipeline(
    text,
    clf,
    nlp,
    targets=None,
    alias_map=None,
    top_k_evidence=3,
    include_neutral=True,
    mixed_margin=0.08,
    text_id=None,
):
    if targets is None:
        targets = suggest_targets(text, nlp)

    out = analyze_text_targets(
        text=text,
        targets=targets,
        clf=clf,
        nlp=nlp,
        alias_map=alias_map,
        top_k_evidence=top_k_evidence,
        include_neutral=include_neutral,
        text_id=text_id,
        mixed_margin=mixed_margin,
    )
    return out

In [32]:
# manual targets
out_manual = run_stance_pipeline(
    text=sample_text_2,
    clf=nli,
    nlp=nlp,
    targets=["Narendra Modi", "BJP", "Congress"],
    alias_map=alias_map_2,
    text_id="manual_targets_demo"
)

print("=== MANUAL TARGETS ===")
print_stance_output(out_manual)

# auto-suggested targets
out_auto = run_stance_pipeline(
    text=sample_text_2,
    clf=nli,
    nlp=nlp,
    targets=None,
    alias_map={
        "Modi": ["Prime Minister Modi", "PM Modi"],
        "BJP": ["BJP-led government", "ruling party"],
        "Congress": ["Congress leaders"]
    },
    text_id="auto_targets_demo"
)

print("\n=== AUTO-SUGGESTED TARGETS ===")
print_stance_output(out_auto)

=== MANUAL TARGETS ===
text_id: manual_targets_demo
TARGET: Narendra Modi
  LABEL: mixed
  SCORES: {'favor': 0.4838, 'against': 0.4741, 'neutral': 0.0421}
  NUM_SENTENCES_USED: 2
  EVIDENCE:
    1. Prime Minister Modi said the reforms would strengthen the economy over time.
    2. Critics argued that Modi had failed to address unemployment and rising inequality.
--------------------------------------------------------------------------------
TARGET: BJP
  LABEL: against
  SCORES: {'favor': 0.3202, 'against': 0.5902, 'neutral': 0.0895}
  NUM_SENTENCES_USED: 2
  EVIDENCE:
    1. Congress leaders said the BJP-led government was ignoring the struggles of ordinary workers.
    2. Supporters of the ruling party responded that the opposition was downplaying welfare delivery and infrastructure gains.
--------------------------------------------------------------------------------
TARGET: Congress
  LABEL: favor
  SCORES: {'favor': 0.739, 'against': 0.237, 'neutral': 0.024}
  NUM_SENTENCES_USED

https://inc.in/key-issues/demonetisation-a-modi-made-disaster

In [33]:
#  fetch and extract the article text from the webpage

!pip -q install requests beautifulsoup4 lxml

import requests
from bs4 import BeautifulSoup

url = "https://inc.in/key-issues/demonetisation-a-modi-made-disaster"

html = requests.get(url, timeout=20).text
print("HTML length:", len(html))

soup = BeautifulSoup(html, "lxml")

# grab paragraph-like text
paras = [p.get_text(" ", strip=True) for p in soup.find_all("p")]
paras = [p for p in paras if p and len(p) > 40]

print("Num candidate paragraphs:", len(paras))
print("\nFirst 5 extracted paragraphs:\n")
for i, p in enumerate(paras[:5], 1):
    print(f"{i}. {p}\n")

HTML length: 92621
Num candidate paragraphs: 17

First 5 extracted paragraphs:

1. On November 8th, 86% of Indiaâs currency was nullified in an effort to clean out âblack moneyâ and âcounterfeit notesâ; this effort resulted in a massive disruption to the existing social, political and economic functions of the worldâs second largest emerging market. All 500 and 1,000 rupee notes were instantaneously voided, and a 50-day period ensued where the population could (ideally) redeem their cancelled cash for freshly issued 2,000 and later 500 rupee notes or deposit them into their respective bank accounts.

2. In the ensuing days after Demonetization, the public in general was hit quite bad, but it was the poor who took the largest share of pain. The poor and the lower middle-classes that constitute the vast majority of the population, simply did not have the access to structural and cultural resources needed to adapt to such shock economics.

3. Even the banks, debuted to do all 

In [34]:
#  clean extracted paragraphs and build article text

def basic_clean(text):
    replacements = {
        "â": "'",
        "â": '"',
        "â": '"',
        "â": "-",
        "â": "-",
        "Â": "",
    }
    for bad, good in replacements.items():
        text = text.replace(bad, good)
    return " ".join(text.split())

clean_paras = [basic_clean(p) for p in paras]
article_text = "\n\n".join(clean_paras)

print("Total cleaned paragraphs:", len(clean_paras))
print("\nFirst 3 cleaned paragraphs:\n")
for i, p in enumerate(clean_paras[:3], 1):
    print(f"{i}. {p}\n")

Total cleaned paragraphs: 17

First 3 cleaned paragraphs:

1. On November 8th, 86% of India's currency was nullified in an effort to clean out "black money" and "counterfeit notes"; this effort resulted in a massive disruption to the existing social, political and economic functions of the world's second largest emerging market. All 500 and 1,000 rupee notes were instantaneously voided, and a 50-day period ensued where the population could (ideally) redeem their cancelled cash for freshly issued 2,000 and later 500 rupee notes or deposit them into their respective bank accounts.

2. In the ensuing days after Demonetization, the public in general was hit quite bad, but it was the poor who took the largest share of pain. The poor and the lower middle-classes that constitute the vast majority of the population, simply did not have the access to structural and cultural resources needed to adapt to such shock economics.

3. Even the banks, debuted to do all the heavy lifting on the ground, 

In [35]:
#  run stance on a few manual targets

targets_india = ["Narendra Modi", "BJP", "Demonetisation"]

alias_map_india = {
    "Narendra Modi": ["Modi", "Prime Minister Modi", "PM Modi"],
    "BJP": ["BJP", "government", "Union government"],
    "Demonetisation": ["Demonetisation", "demonetization", "demonetised", "demonetized"]
}

out_india = run_stance_pipeline(
    text=article_text,
    clf=nli,
    nlp=nlp,
    targets=targets_india,
    alias_map=alias_map_india,
    top_k_evidence=3,
    include_neutral=True,
    mixed_margin=0.08,
    text_id="india_demo_demonetisation"
)

print_stance_output(out_india)

text_id: india_demo_demonetisation
TARGET: Narendra Modi
  LABEL: against
  SCORES: {'favor': 0.3856, 'against': 0.5698, 'neutral': 0.0446}
  NUM_SENTENCES_USED: 1
  EVIDENCE:
    1. Even after the supportive mainstream media declared demonetisation a failure, PM Modi has still not been able to bring himself to condole with their bereaved families or pay them any compensation for the loss of in many cases, their primary breadwinners.
--------------------------------------------------------------------------------
TARGET: BJP
  LABEL: mixed
  SCORES: {'favor': 0.4079, 'against': 0.4048, 'neutral': 0.1873}
  NUM_SENTENCES_USED: 2
  EVIDENCE:
    1. Demonetisation was a colossal and completely avoidable failure and the largest government-abetted money laundering scheme in history.
    2. Even the banks, debuted to do all the heavy lifting on the ground, weren't kept in the loop; ill-equipped for the crisis and unable to make sense of an outlandish government order, they still managed to d

In [36]:
#  tighten BJP aliases and compare outputs

alias_map_india_tight = {
    "Narendra Modi": ["Modi", "Prime Minister Modi", "PM Modi"],
    "BJP": ["BJP", "Bharatiya Janata Party"],
    "Demonetisation": ["Demonetisation", "demonetization", "demonetised", "demonetized"]
}

out_india_tight = run_stance_pipeline(
    text=article_text,
    clf=nli,
    nlp=nlp,
    targets=targets_india,
    alias_map=alias_map_india_tight,
    top_k_evidence=3,
    include_neutral=True,
    mixed_margin=0.08,
    text_id="india_demo_demonetisation_tight_alias"
)

print_stance_output(out_india_tight)

text_id: india_demo_demonetisation_tight_alias
TARGET: Narendra Modi
  LABEL: against
  SCORES: {'favor': 0.3856, 'against': 0.5698, 'neutral': 0.0446}
  NUM_SENTENCES_USED: 1
  EVIDENCE:
    1. Even after the supportive mainstream media declared demonetisation a failure, PM Modi has still not been able to bring himself to condole with their bereaved families or pay them any compensation for the loss of in many cases, their primary breadwinners.
--------------------------------------------------------------------------------
TARGET: BJP
  LABEL: no_evidence
  SCORES: {}
  NUM_SENTENCES_USED: 0
  EVIDENCE:
--------------------------------------------------------------------------------
TARGET: Demonetisation
  LABEL: against
  SCORES: {'favor': 0.26, 'against': 0.5572, 'neutral': 0.1828}
  NUM_SENTENCES_USED: 7
  EVIDENCE:
    1. The demonetisation move represents not just a faulty economic policy but also holds high potential for indiscriminate state surveillance, violation of privacy 

In [ ]:
# registry


In [38]:
#registry-aware target generation + final pipeline wrapper

import re

# ---- Target resolution helpers ----
def resolve_target(input_target, registry):
    """
    Map raw input target like 'Modi' to canonical target like 'Narendra Modi'.
    If no match, return input_target unchanged.
    """
    if registry is None:
        return input_target

    # exact canonical key
    if input_target in registry:
        return input_target

    low = input_target.lower().strip()
    for canon, meta in registry.items():
        aliases = meta.get("aliases", [])
        if low == canon.lower():
            return canon
        for a in aliases:
            if low == a.lower():
                return canon

    return input_target

def get_aliases_for_target(target, registry):
    """
    Return [canonical] + aliases if present in registry, else [target].
    """
    canon = resolve_target(target, registry)
    if registry is None or canon not in registry:
        return [canon]
    aliases = registry[canon].get("aliases", [])
    # preserve order, dedupe
    vals = [canon] + aliases
    seen = set()
    out = []
    for x in vals:
        xl = x.lower()
        if xl not in seen:
            seen.add(xl)
            out.append(x)
    return out

# ---- Safer matching ----
def compile_alias_patterns(names):
    patterns = []
    for name in names:
        pat = re.compile(r"\b" + re.escape(name) + r"\b", flags=re.IGNORECASE)
        patterns.append(pat)
    return patterns

def sentence_mentions_any(sent, names):
    patterns = compile_alias_patterns(names)
    return any(p.search(sent) for p in patterns)

# ---- Target suggestion: NER + issue aliases from registry ----
def suggest_targets_from_registry(text, nlp, registry, country_filter=None):
    suggested = []

    # NER entities
    doc = nlp(text)
    ner_candidates = []
    for ent in doc.ents:
        if ent.label_ in {"PERSON", "ORG"}:
            ner_candidates.append(ent.text.strip())

    # resolve NER candidates to canonical targets
    for cand in ner_candidates:
        canon = resolve_target(cand, registry)
        if canon in registry:
            if country_filter is None or registry[canon].get("country") == country_filter:
                suggested.append(canon)

    # issue keyword scan from registry aliases
    text_low = text.lower()
    for canon, meta in registry.items():
        if country_filter is not None and meta.get("country") != country_filter:
            continue
        for alias in meta.get("aliases", []):
            if alias.lower() in text_low:
                suggested.append(canon)
                break

    # dedupe preserve order
    seen = set()
    out = []
    for x in suggested:
        if x.lower() not in seen:
            seen.add(x.lower())
            out.append(x)
    return out

# ---- Final wrapper ----
def run_stance_pipeline_registry(
    text,
    clf,
    nlp,
    registry=None,
    targets=None,                 # user-provided raw or canonical targets
    do_ner_target_suggestion=True,
    country_filter=None,
    top_k_evidence=3,
    include_neutral=True,
    mixed_margin=0.08,
    text_id=None,
    retrieval_mode = "expanded",
):
    # choose targets
    requested_targets = targets[:] if targets is not None else []

    auto_targets = []
    if do_ner_target_suggestion:
        auto_targets = suggest_targets_from_registry(text, nlp, registry or {}, country_filter=country_filter)

    # union while preserving order
    combined_targets = []
    seen = set()
    for t in requested_targets + auto_targets:
        canon = resolve_target(t, registry or {})
        if canon.lower() not in seen:
            seen.add(canon.lower())
            combined_targets.append(canon)

    results = []
    sentences = split_sentences(text, nlp)

    for target in combined_targets:
        # names = get_aliases_for_target(target, registry or {})

        # matched_sentences = [s for s in sentences if sentence_mentions_any(s, names)]
        # CHANGE: adding retrival mode, if string MODI may not match with BJP etc.
        if retrieval_mode == "strict":
            names = get_aliases_for_target(target, registry)
        elif retrieval_mode == "expanded":
            names = get_aliases_for_target(target, registry) + get_related_names(target, registry)

        matched_sentences = [s for s in sentences if sentence_mentions_any(s, names)]

        if not matched_sentences:
            results.append({
                "target": target,
                "label": "no_evidence",
                "scores": {},
                "evidence": [],
                "num_sentences_used": 0
            })
            continue

        scored = []
        for sent in matched_sentences:
            labels = [
                f"This text supports {target}.",
                f"This text opposes {target}.",
            ]
            if include_neutral:
                labels.append(f"This text is neutral toward {target}.")
            out = clf(sent, labels, multi_label=False)
            score_map = dict(zip(out["labels"], out["scores"]))
            scored.append((sent, score_map))

        support_key = f"This text supports {target}."
        oppose_key = f"This text opposes {target}."

        mean_scores = {
            "favor": sum(x[1].get(support_key, 0.0) for x in scored) / len(scored),
            "against": sum(x[1].get(oppose_key, 0.0) for x in scored) / len(scored),
        }

        if include_neutral:
            neutral_key = f"This text is neutral toward {target}."
            mean_scores["neutral"] = sum(x[1].get(neutral_key, 0.0) for x in scored) / len(scored)

        sorted_scores = sorted(mean_scores.items(), key=lambda x: x[1], reverse=True)
        best_label, best_score = sorted_scores[0]
        second_label, second_score = sorted_scores[1]

        if abs(best_score - second_score) < mixed_margin:
            final_label = "mixed"
            top_two = {best_label, second_label}
            evidence_scored = []
            for sent, score_map in scored:
                sent_score = 0.0
                if "favor" in top_two:
                    sent_score = max(sent_score, score_map.get(support_key, 0.0))
                if "against" in top_two:
                    sent_score = max(sent_score, score_map.get(oppose_key, 0.0))
                if include_neutral and "neutral" in top_two:
                    sent_score = max(sent_score, score_map.get(neutral_key, 0.0))
                evidence_scored.append((sent, sent_score))
            evidence = sorted(evidence_scored, key=lambda x: x[1], reverse=True)[:top_k_evidence]
            evidence_text = [x[0] for x in evidence]
        else:
            final_label = best_label
            winner_key = support_key if final_label == "favor" else oppose_key if final_label == "against" else neutral_key
            evidence = sorted(scored, key=lambda x: x[1].get(winner_key, 0.0), reverse=True)[:top_k_evidence]
            evidence_text = [x[0] for x in evidence]

        results.append({
            "target": target,
            "label": final_label,
            "scores": mean_scores,
            "evidence": evidence_text,
            "num_sentences_used": len(matched_sentences)
        })

    return {
        "text_id": text_id,
        "requested_targets": requested_targets,
        "auto_targets": auto_targets,
        "results": results
    }

In [39]:
#  fetch NDTV article and run registry-aware pipeline

import requests
from bs4 import BeautifulSoup

url = "https://www.ndtv.com/india-news/wont-be-any-discrimination-pms-guarantee-on-womens-quota-delimitation-11365855"
html = requests.get(url, timeout=20).text
soup = BeautifulSoup(html, "lxml")

paras = [p.get_text(" ", strip=True) for p in soup.find_all("p")]
paras = [p for p in paras if p and len(p) > 40]

def basic_clean(text):
    replacements = {
        "â": "'",
        "â": '"',
        "â": '"',
        "â": "-",
        "â": "-",
        "Â": "",
    }
    for bad, good in replacements.items():
        text = text.replace(bad, good)
    return " ".join(text.split())

clean_paras = [basic_clean(p) for p in paras]
ndtv_text = "\n\n".join(clean_paras)

# test 1: auto target generation
out_ndtv_auto = run_stance_pipeline_registry(
    text=ndtv_text,
    clf=nli,
    nlp=nlp,
    registry=TARGET_REGISTRY,
    targets=None,
    do_ner_target_suggestion=True,
    country_filter="india",
    top_k_evidence=2,
    include_neutral=True,
    mixed_margin=0.08,
    text_id="ndtv_womens_quota_auto"
)

print("AUTO TARGETS:", out_ndtv_auto["auto_targets"])
print_stance_output({"text_id": out_ndtv_auto["text_id"], "results": out_ndtv_auto["results"]})

# with user targets + auto suggestions
user_targets = ["Modi", "Congress", "Delimitation", "Women's Reservation Bill"]
out_ndtv_user = run_stance_pipeline_registry(
    text=ndtv_text,
    clf=nli,
    nlp=nlp,
    registry=TARGET_REGISTRY,
    targets=user_targets,
    do_ner_target_suggestion=True,
    country_filter="india",
    top_k_evidence=2,
    include_neutral=True,
    mixed_margin=0.08,
    text_id="ndtv_womens_quota_user_plus_auto"
)

print("REQUESTED TARGETS:", out_ndtv_user["requested_targets"])
print("AUTO TARGETS:", out_ndtv_user["auto_targets"])
print_stance_output({"text_id": out_ndtv_user["text_id"], "results": out_ndtv_user["results"]})

AUTO TARGETS: []
text_id: ndtv_womens_quota_auto
REQUESTED TARGETS: ['Modi', 'Congress', 'Delimitation', "Women's Reservation Bill"]
AUTO TARGETS: []
text_id: ndtv_womens_quota_user_plus_auto
TARGET: Narendra Modi
  LABEL: no_evidence
  SCORES: {}
  NUM_SENTENCES_USED: 0
  EVIDENCE:
--------------------------------------------------------------------------------
TARGET: Congress
  LABEL: no_evidence
  SCORES: {}
  NUM_SENTENCES_USED: 0
  EVIDENCE:
--------------------------------------------------------------------------------
TARGET: Delimitation
  LABEL: no_evidence
  SCORES: {}
  NUM_SENTENCES_USED: 0
  EVIDENCE:
--------------------------------------------------------------------------------
TARGET: Women's Reservation Bill
  LABEL: no_evidence
  SCORES: {}
  NUM_SENTENCES_USED: 0
  EVIDENCE:
--------------------------------------------------------------------------------


In [40]:
#  inspect what was actually extracted from the NDTV page

print("Num paragraphs:", len(clean_paras))
print("\nFirst 10 extracted paragraphs:\n")
for i, p in enumerate(clean_paras[:10], 1):
    print(f"{i}. {p[:500]}\n")

print("\nCombined text preview:\n")
print(ndtv_text[:3000])

Num paragraphs: 2

First 10 extracted paragraphs:

1. Reference #18.c43f2e17.1776342705.60c367d1

2. https://errors.edgesuite.net/18.c43f2e17.1776342705.60c367d1


Combined text preview:

Reference #18.c43f2e17.1776342705.60c367d1

https://errors.edgesuite.net/18.c43f2e17.1776342705.60c367d1


In [41]:
#  retry fetch with browser headers

import requests
from bs4 import BeautifulSoup

url = "https://www.ndtv.com/india-news/wont-be-any-discrimination-pms-guarantee-on-womens-quota-delimitation-11365855"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "en-US,en;q=0.9",
}

resp = requests.get(url, headers=headers, timeout=20)
print("Status:", resp.status_code)
print("Final URL:", resp.url)
print(resp.text[:1000])

Status: 403
Final URL: https://www.ndtv.com/india-news/wont-be-any-discrimination-pms-guarantee-on-womens-quota-delimitation-11365855
<HTML><HEAD>
<TITLE>Access Denied</TITLE>
</HEAD><BODY>
<H1>Access Denied</H1>
 
You don't have permission to access "http&#58;&#47;&#47;www&#46;ndtv&#46;com&#47;india&#45;news&#47;wont&#45;be&#45;any&#45;discrimination&#45;pms&#45;guarantee&#45;on&#45;womens&#45;quota&#45;delimitation&#45;11365855" on this server.<P>
Reference&#32;&#35;18&#46;c43f2e17&#46;1776342902&#46;60c9f3fb
<P>https&#58;&#47;&#47;errors&#46;edgesuite&#46;net&#47;18&#46;c43f2e17&#46;1776342902&#46;60c9f3fb</P>
</BODY>
</HTML>



In [42]:
manual_article_text = """

Prime Minister Narendra Modi on Thursday offered guarantees and rebukes to the opposition - with a jibe for the Congress, though he didn't name the party - as they protested against three contentious bills tabled in Parliament. "We look at India as one... not in parts," he declared.

On the delimitation bill - the redrawing of parliamentary and constituency boundaries - he said: "I give my guarantee... no injustice will be done to any state, from east to west, north to south."

The guarantee followed repeated agitations by southern states who fear redrawing constituency boundaries will translate into losing parliamentary seats (and heft at the centre) to northern, Hindi-speaking states - like Uttar Pradesh - that are seen as BJP strongholds.

"The Opposition is spreading confusion regarding women's reservation. The number of seats in the Lok Sabha and State Legislative Assemblies will increase by 50 per cent. The states' proportional representation in the Lok Sabha will remain at its current level," top governmmet sources told NDTV.

And responding to criticism of the bill to give women 33 per cent reservation in the expanded (by nearly 50 per cent) Lok Sabha, he said: Those who opposed this in the past... they were not forgiven by women of the country and they ended up badly in elections that followed."

"Let all of us not miss this important opportunity to give reservation to women," he said, "I have come to appeal to you - do not see this from a political lens, this is in national interest."

Mocking opposition parties who, in the past, accused him of taking credit for welfare schemes introduced by earlier governments, he laughed: "You can take credit. I am willing to give ads with photos of whoever... giving a 'blank cheque' to take the credit for the women's quota."

Reservation for women lawmakers, he said, was something that should have been enforced 25-30 years ago. "(Had we) implemented it then, today we would have been a mature country."

A fiercely critical opposition has said proposals to redraw constituency boundaries and increase Lok Sabha seats are "flawed" - Congress chief Mallikarjun Kharge called it "an assault on our democracy" - and had been bundled with the women's bill to force it through parliament.

The delimitation exercise is a longstanding flashpoint between the centre and southern states, one that rears its head periodically and has done so again ahead of the election in Tamil Nadu this month. The southern state is a particularly vocal opponent of delimitation, arguing that it, like its neighbours, will be penalised for having successfully controlled population levels.

All three bills - the third increases the number of Lok Sabha seats to 850 - were tabled in Parliament this afternoon. Two were introduced by Law Minister Arjun Meghwal.
"""

In [43]:
manual_article_text = basic_clean(manual_article_text)

print("Text preview:\n")
print(manual_article_text[:1500])

# auto targets only
out_manual_auto = run_stance_pipeline_registry(
    text=manual_article_text,
    clf=nli,
    nlp=nlp,
    registry=TARGET_REGISTRY,
    targets=None,
    do_ner_target_suggestion=True,
    country_filter="india",
    top_k_evidence=2,
    include_neutral=True,
    mixed_margin=0.08,
    text_id="manual_article_auto"
)

print("\nAUTO TARGETS:", out_manual_auto["auto_targets"])
print_stance_output({
    "text_id": out_manual_auto["text_id"],
    "results": out_manual_auto["results"]
})

Text preview:

Prime Minister Narendra Modi on Thursday offered guarantees and rebukes to the opposition - with a jibe for the Congress, though he didn't name the party - as they protested against three contentious bills tabled in Parliament. "We look at India as one... not in parts," he declared. On the delimitation bill - the redrawing of parliamentary and constituency boundaries - he said: "I give my guarantee... no injustice will be done to any state, from east to west, north to south." The guarantee followed repeated agitations by southern states who fear redrawing constituency boundaries will translate into losing parliamentary seats (and heft at the centre) to northern, Hindi-speaking states - like Uttar Pradesh - that are seen as BJP strongholds. "The Opposition is spreading confusion regarding women's reservation. The number of seats in the Lok Sabha and State Legislative Assemblies will increase by 50 per cent. The states' proportional representation in the Lok Sabha will rem

You can use outlet/source leaning as a label for article-level ideology or source-style bias, because the Hindi bias paper and other bias work already frame the problem as article/news bias detection rather than target stance.

BART-MNLI

You are using it as an NLI engine:

premise = text chunk
hypothesis = “This text supports X.”

That is excellent for zero-shot, but if you fine-tune it, you are still stuck in the premise-hypothesis formulation, which means:

more preprocessing
more inference calls
more awkward deployment

The model card itself describes it as a ready-made zero-shot classifier through NLI premise/hypothesis scoring.

RoBERTa

For supervised stance, you can train directly as:

input: text + target
output: favor / against / neutral

That is simpler, faster, and cleaner to package.

So my recommendation stays:

BART-MNLI = zero-shot baseline + explainable scorer
RoBERTa-base = first supervised fine-tuned model

## Small experiment

OpIndia for the pro-BJP / right-leaning pole. OpIndia is widely described as openly right-leaning and supportive of BJP/Hindutva narratives, and has also faced repeated credibility criticism.
The Wire for the anti-government / left-leaning pole. Media Bias/Fact Check rates it as generally factual and left-leaning, and it is widely known for critical coverage of the Modi government.
The Indian Express as the more mainstream middle anchor. It is not perfectly “centrist,” but it is much closer to a conventional national newspaper than the other two and gives you a stabilizing comparison point. MBFC rates it left-center rather than hard-left.

In [46]:

TARGET_REGISTRY = {
    # India
    "Narendra Modi": {
        "aliases": ["Narendra Modi", "Modi", "PM Modi", "Prime Minister Modi"],
        "kind": "person",
        "country": "india",
    },
    "Mallikarjun Kharge": {
        "aliases": ["Mallikarjun Kharge", "Kharge"],
        "kind": "person",
        "country": "india",
    },
    "Congress": {
        "aliases": ["Congress", "INC", "Indian National Congress"],
        "kind": "party",
        "country": "india",
    },
    "BJP": {
        "aliases": ["BJP", "Bharatiya Janata Party"],
        "kind": "party",
        "country": "india",
    },
    "Demonetisation": {
        "aliases": ["Demonetisation", "demonetization", "demonetised", "demonetized", "notebandi"],
        "kind": "issue",
        "country": "india",
    },
    "Delimitation": {
        "aliases": ["delimitation", "constituency boundaries", "redrawing constituency boundaries"],
        "kind": "issue",
        "country": "india",
    },
    "Women's Reservation Bill": {
        "aliases": ["women's reservation bill", "women reservation", "women's quota", "women quota", "33 per cent reservation"],
        "kind": "issue",
        "country": "india",
    },
    # US
    "Donald Trump": {
        "aliases": ["Donald Trump", "Trump", "President Trump"],
        "kind": "person",
        "country": "us",
    },
    "Joe Biden": {
        "aliases": ["Joe Biden", "Biden", "President Biden"],
        "kind": "person",
        "country": "us",
    },
    "Bernie Sanders": {
        "aliases": ["Bernie Sanders", "Bernie", "Sanders"],
        "kind": "person",
        "country": "us",
    },
    "Republican Party": {
        "aliases": ["Republican Party", "Republicans", "GOP"],
        "kind": "party",
        "country": "us",
    },
    "Democratic Party": {
        "aliases": ["Democratic Party", "Democrats", "Dems"],
        "kind": "party",
        "country": "us",
    },
}

In [47]:
#  scrape 3 outlets on the same event window and run stance analysis

!pip -q install trafilatura requests pandas

import requests
import trafilatura
import pandas as pd

# 1) article set
articles = [
    {
        "outlet": "OpIndia",
        "url": "https://www.opindia.com/2026/04/opposition-for-optics-not-principle-congress-sp-and-allies-cry-haste-on-womens-reservation-bill-despite-earlier-demanding-immediate-rollout-hypocrisy-exposed/"
    },
    {
        "outlet": "The Wire",
        "url": "https://m.thewire.in/article/politics/opposition-india-bloc-to-oppose-delimitation-bill"
    },
    {
        "outlet": "Indian Express",
        "url": "https://indianexpress.com/article/india/womens-quota-pm-modi-seeks-support-opp-says-no-details-on-delimitation-10633168/"
    }
]

# 2) fixed target list for comparison
targets = [
    "Narendra Modi",
    "Congress",
    "Mallikarjun Kharge",
    "Delimitation",
    "Women's Reservation Bill"
]

# 3) helper to fetch text
def fetch_article_text(url):
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept-Language": "en-US,en;q=0.9",
    }
    resp = requests.get(url, headers=headers, timeout=25)
    html = resp.text
    text = trafilatura.extract(html, include_links=False, include_comments=False)
    return {
        "status_code": resp.status_code,
        "final_url": resp.url,
        "html_len": len(html),
        "text": text if text else ""
    }

# 4) scrape + run stance
rows = []
full_outputs = {}

for art in articles:
    fetch = fetch_article_text(art["url"])
    text = basic_clean(fetch["text"]) if fetch["text"] else ""

    print(f"\n=== {art['outlet']} ===")
    print("status:", fetch["status_code"])
    print("final_url:", fetch["final_url"])
    print("text_len:", len(text))
    print("preview:", text[:500], "\n")

    if len(text) < 200:
        print("Skipping due to insufficient extracted text.\n")
        continue

    out = run_stance_pipeline_registry(
        text=text,
        clf=nli,
        nlp=nlp,
        registry=TARGET_REGISTRY,
        targets=targets,
        do_ner_target_suggestion=False,   # fixed targets for cleaner comparison
        country_filter="india",
        top_k_evidence=2,
        include_neutral=True,
        mixed_margin=0.08,
        text_id=art["outlet"]
    )

    full_outputs[art["outlet"]] = out

    for res in out["results"]:
        row = {
            "outlet": art["outlet"],
            "target": res["target"],
            "label": res["label"],
            "favor_score": res["scores"].get("favor", None) if res["scores"] else None,
            "against_score": res["scores"].get("against", None) if res["scores"] else None,
            "neutral_score": res["scores"].get("neutral", None) if res["scores"] else None,
            "num_sentences_used": res["num_sentences_used"],
            "evidence_1": res["evidence"][0] if len(res["evidence"]) > 0 else "",
            "evidence_2": res["evidence"][1] if len(res["evidence"]) > 1 else "",
        }
        rows.append(row)

results_df = pd.DataFrame(rows)

print("\n=== Compact comparison table ===")
display(results_df)

print("\n=== Label cross-tab ===")
if len(results_df) > 0:
    display(pd.crosstab(results_df["target"], results_df["outlet"] + " | " + results_df["label"]))

print("\n=== Mean stance scores by outlet ===")
if len(results_df) > 0:
    display(results_df.groupby("outlet")[["favor_score", "against_score", "neutral_score"]].mean())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 21.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 20.1 MB/s eta 0:00:00

=== OpIndia ===
status: 200
final_url: https://www.opindia.com/2026/04/opposition-for-optics-not-principle-congress-sp-and-allies-cry-haste-on-womens-reservation-bill-despite-earlier-demanding-immediate-rollout-hypocrisy-exposed/
text_len: 11830
preview: On 3rd April (Friday), Union Minister of Parliamentary Affairs Kiren Rijiju informed that the Women Reservation Bill of the Nari Shakti Vandan Adhiniyam is going to be discussed in a special session of Parliament from 16th to 18th April. “We are convening the Parliament on 16th April. We will take up the Women’s Reservation Bill then. Empowerment of women is our commitment. We must come together for the empowerme

,outlet,target,label,favor_score,against_score,neutral_score,num_sentences_used,evidence_1,evidence_2
0,OpIndia,Narendra Modi,against,0.362690,0.617537,0.019772,4,The Congress party held 15 press conferences i...,“We won’t back down on our demand: Immediate i...
1,OpIndia,Congress,favor,0.601899,0.378032,0.020069,11,Congress demands instant implementation of the...,"— Congress (@INCIndia) September 29, 2023 We w..."
2,OpIndia,Mallikarjun Kharge,against,0.211707,0.758869,0.029425,3,Kharge was countered by Leader of the House an...,"Mallikarjun Kharge, the leader of the oppositi..."
3,OpIndia,Delimitation,against,0.345585,0.549293,0.105122,12,From wanting swift implementation to resisting...,"The All India Trinamool Congress Lok Sabha MP,..."
4,OpIndia,Women's Reservation Bill,against,0.330687,0.599087,0.070225,3,"Therefore, the bill could not be executed even...",The Congress party held 15 press conferences i...
5,The Wire,Narendra Modi,against,0.122587,0.835563,0.041850,1,"INDIA Bloc to Oppose Delimitation Bill, Says G...",
6,The Wire,Congress,favor,0.868887,0.082145,0.048969,3,Congress MP Jairam Ramesh passed in 2023 shoul...,The meeting was attended by leaders from the C...
7,The Wire,Mallikarjun Kharge,favor,0.732407,0.235158,0.032434,3,Following the meeting of all opposition partie...,Kharge said that women’s reservation has been ...
8,The Wire,Delimitation,against,0.338793,0.627143,0.034063,8,"INDIA Bloc to Oppose Delimitation Bill, Says G...","Opposition parties held a joint meeting, follo..."
9,The Wire,Women's Reservation Bill,against,0.006970,0.975896,0.017134,1,"INDIA Bloc to Oppose Delimitation Bill, Says G...",



=== Label cross-tab ===


col_0,Indian Express | against,Indian Express | no_evidence,OpIndia | against,OpIndia | favor,The Wire | against,The Wire | favor
target,,,,,,
Congress,0,1,0,1,0,1
Delimitation,1,0,1,0,1,0
Mallikarjun Kharge,0,1,1,0,0,1
Narendra Modi,1,0,1,0,1,0
Women's Reservation Bill,0,1,1,0,1,0



=== Mean stance scores by outlet ===


,favor_score,against_score,neutral_score
outlet,,,
Indian Express,0.168849,0.765976,0.065175
OpIndia,0.370514,0.580564,0.048923
The Wire,0.413929,0.551181,0.034890


In [48]:
results_df[["outlet", "target", "label", "favor_score", "against_score", "neutral_score", "num_sentences_used"]]

,outlet,target,label,favor_score,against_score,neutral_score,num_sentences_used
0,OpIndia,Narendra Modi,against,0.362690,0.617537,0.019772,4
1,OpIndia,Congress,favor,0.601899,0.378032,0.020069,11
2,OpIndia,Mallikarjun Kharge,against,0.211707,0.758869,0.029425,3
3,OpIndia,Delimitation,against,0.345585,0.549293,0.105122,12
4,OpIndia,Women's Reservation Bill,against,0.330687,0.599087,0.070225,3
5,The Wire,Narendra Modi,against,0.122587,0.835563,0.041850,1
6,The Wire,Congress,favor,0.868887,0.082145,0.048969,3
7,The Wire,Mallikarjun Kharge,favor,0.732407,0.235158,0.032434,3
8,The Wire,Delimitation,against,0.338793,0.627143,0.034063,8
9,The Wire,Women's Reservation Bill,against,0.006970,0.975896,0.017134,1


In [49]:
results_df.pivot_table(
    index="target",
    columns="outlet",
    values=["label", "favor_score", "against_score", "neutral_score"],
    aggfunc="first"
)

against_score                        favor_score  \
outlet                   Indian Express   OpIndia  The Wire Indian Express   
target                                                                       
Congress                            NaN  0.378032  0.082145            NaN   
Delimitation                   0.817472  0.549293  0.627143       0.088176   
Mallikarjun Kharge                  NaN  0.758869  0.235158            NaN   
Narendra Modi                  0.714479  0.617537  0.835563       0.249522   
Women's Reservation Bill            NaN  0.599087  0.975896            NaN   

                                                      label                    \
outlet                     OpIndia  The Wire Indian Express  OpIndia The Wire   
target                                                                          
Congress                  0.601899  0.868887    no_evidence    favor    favor   
Delimitation              0.345585  0.338793        against  against  against   
Mallikarjun Kharge        0.211707  0.732407    no_evidence  against    favor   
Narendra Modi             0.362690  0.122587        against  against  against   
Women's Reservation Bill  0.330687  0.006970    no_evidence  against  against   

                          neutral_score                      
outlet                   Indian Express   OpIndia  The Wire  
target                                                       
Congress                            NaN  0.020069  0.048969  
Delimitation                   0.094352  0.105122  0.034063  
Mallikarjun Kharge                  NaN  0.029425  0.032434  
Narendra Modi                  0.035999  0.019772  0.041850  
Women's Reservation Bill            NaN  0.070225  0.017134